In [6]:
# ============================================================
# Railway Dataset — Annotation Verification & Class Correction (Google Colab)
# ============================================================
# Run this in a Colab notebook cell.
#
# Workflow per image:
#   1. Image loads with color-coded boxes drawn per class.
#   2. The class-id table below is editable — fix any wrong class numbers.
#      A class (e.g. "Bolt" or "Wheel") can have ANY number of boxes —
#      the table always shows one row per actual box in the label file,
#      never one row per class.
#   3. Click "Preview edits" to redraw the image with your corrected classes
#      (nothing is saved yet).
#   4. Click "Approve & Save" to write the corrected label file and copy the
#      image + label pair to the Main Dataset, using matching filenames
#      (imgXYZ.jpg <-> imgXYZ.txt).
#
# Install deps (uncomment if needed):
# !pip install -q gradio pandas

import os
import glob
import shutil
import zipfile

import gradio as gr
import pandas as pd
from PIL import Image, ImageDraw

# ------------------------------------------------------------
# STEP 0: Mount Drive
# ------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# ------------------------------------------------------------
# STEP 1: Unzip set2.zip from Drive -> LOCAL disk (/content), not Drive
# ------------------------------------------------------------
SRC_ZIP = "/content/drive/MyDrive/ASN Demo Video/Railway Test Video/Datsets Arrangement/set2.zip"
IMG_EXTRACT_DIR = "/content/Set2"

os.makedirs(IMG_EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(SRC_ZIP, 'r') as z:
    z.extractall(IMG_EXTRACT_DIR)
print(f"Extracted images to {IMG_EXTRACT_DIR}")

# ------------------------------------------------------------
# STEP 2: Unzip /content/Set2an.zip (annotations) -> local disk
# ------------------------------------------------------------
ANN_ZIP = "/content/Set2an.zip"
ANN_EXTRACT_DIR = "/content/Set2an"

os.makedirs(ANN_EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ANN_ZIP, 'r') as z:
    z.extractall(ANN_EXTRACT_DIR)
print(f"Extracted annotations to {ANN_EXTRACT_DIR}")

# ------------------------------------------------------------
# Destination folders on Drive (Main Dataset)
# ------------------------------------------------------------
DEST_IMG_DIR = "/content/drive/MyDrive/ASN Demo Video/Railway Test Video/Main Dataset/images"
DEST_LBL_DIR = "/content/drive/MyDrive/ASN Demo Video/Railway Test Video/Main Dataset/labels"
os.makedirs(DEST_IMG_DIR, exist_ok=True)
os.makedirs(DEST_LBL_DIR, exist_ok=True)

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

# ------------------------------------------------------------
# Class names + one distinct color per class
# ------------------------------------------------------------
CLASS_NAMES = {
    0: "Bogie",
    1: "Spring",
    2: "Bearing",
    3: "Bolt",
    4: "Handbrake wheel",
    5: "Wheel",
    6: "Empod",
    7: "Air Reservoir",
}

CLASS_COLORS = {
    0: (230, 25, 75),    # red        - Bogie
    1: (60, 180, 75),    # green      - Spring
    2: (255, 225, 25),   # yellow     - Bearing
    3: (0, 130, 200),    # blue       - Bolt
    4: (245, 130, 48),   # orange     - Handbrake wheel
    5: (145, 30, 180),   # purple     - Wheel
    6: (70, 240, 240),   # cyan       - Empod
    7: (240, 50, 230),   # magenta    - Air Reservoir
}
DEFAULT_COLOR = (255, 255, 255)  # fallback for any unexpected class id

VALID_CLASS_IDS = set(CLASS_NAMES.keys())

# Maximum number of boxes (rows) supported per image. Classes can repeat any
# number of times within this limit (e.g. Bolt x10 + Wheel x10 + ... is fine
# as long as the total box count for one image stays <= this).
MAX_BOXES_PER_IMAGE = 25


def find_images(root):
    files = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.lower().endswith(IMG_EXTS):
                files.append(os.path.join(dirpath, f))
    return sorted(files)


def find_label_path(img_path, ann_dir):
    """Find the .txt label file matching this image's base filename.
    Prints a warning if more than one candidate is found, since silently
    picking matches[0] could load the wrong file when duplicate basenames
    exist in different subfolders."""
    base = os.path.splitext(os.path.basename(img_path))[0]
    matches = glob.glob(os.path.join(ann_dir, '**', base + '.txt'), recursive=True)
    if len(matches) > 1:
        print(f"[WARN] Multiple label files matched for '{base}': {matches}. Using {matches[0]}.")
    return matches[0] if matches else None


images = find_images(IMG_EXTRACT_DIR)
if not images:
    print("WARNING: No images found in", IMG_EXTRACT_DIR)

state = {"idx": 0}
# Holds unsaved edits per image path: {img_path: [[cls, xc, yc, bw, bh], ...]}
edited_boxes = {}


def parse_label_file(label_path):
    """Returns list of [cls_id(int), xc, yc, bw, bh] floats.
    Every non-empty, well-formed line becomes its own box — a class
    (e.g. Bolt, Wheel) can appear on any number of lines/boxes."""
    boxes = []
    if label_path and os.path.exists(label_path):
        raw_lines = open(label_path).read().strip().splitlines()
        skipped = 0
        for line in raw_lines:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) >= 5:
                try:
                    cls_id = int(float(parts[0]))
                    xc, yc, bw, bh = map(float, parts[1:5])
                    boxes.append([cls_id, xc, yc, bw, bh])
                except ValueError:
                    skipped += 1
            else:
                skipped += 1
        if skipped:
            print(f"[WARN] {skipped} malformed line(s) skipped in {label_path}")
    return boxes


def boxes_for_image(img_path):
    """Return current boxes for an image: edited version if present, else parsed from file."""
    if img_path in edited_boxes:
        return edited_boxes[img_path]
    label_path = find_label_path(img_path, ANN_EXTRACT_DIR)
    boxes = parse_label_file(label_path)
    # Debug trace: confirms raw file line count vs parsed box count so any
    # future mismatch (e.g. lines silently dropped) is visible in logs.
    if label_path:
        raw_count = len([l for l in open(label_path).read().strip().splitlines() if l.strip()])
        if raw_count != len(boxes):
            print(f"[DEBUG] {os.path.basename(img_path)}: raw lines={raw_count}, parsed boxes={len(boxes)}")
    return boxes


def boxes_to_df(boxes):
    """boxes: list of [cls_id, xc, yc, bw, bh]. Adds a stable 'id' so rows stay
    identifiable even when several boxes share the same class (e.g. 3 Bolts,
    2 Wheels) — one row per box, never one row per class."""
    if not boxes:
        return pd.DataFrame(columns=["id", "class_id", "class_name", "x_center", "y_center", "width", "height"])
    rows = []
    for i, (cls_id, xc, yc, bw, bh) in enumerate(boxes):
        rows.append([i, cls_id, CLASS_NAMES.get(cls_id, "UNKNOWN"), xc, yc, bw, bh])
    return pd.DataFrame(rows, columns=["id", "class_id", "class_name", "x_center", "y_center", "width", "height"])


def df_to_boxes(df):
    """Returns list of [cls_id, xc, yc, bw, bh], in the same row order as the table
    (the 'id' column is display-only and not required to be sequential)."""
    boxes = []
    if df is None:
        return boxes
    for _, row in df.iterrows():
        try:
            cls_id = int(float(row["class_id"]))
            xc = float(row["x_center"])
            yc = float(row["y_center"])
            bw = float(row["width"])
            bh = float(row["height"])
            boxes.append([cls_id, xc, yc, bw, bh])
        except (ValueError, TypeError):
            continue  # skip malformed rows
    return boxes


def sync_class_names(df):
    """Refresh the class_name hint column to match whatever class_id was typed,
    so it never goes stale — critical when multiple boxes share a class and you're
    editing one of several rows with the same original label."""
    if df is None or df.empty:
        return df
    df = df.copy()
    df["class_name"] = df["class_id"].apply(
        lambda c: CLASS_NAMES.get(int(float(c)), f"UNKNOWN({c})") if pd.notna(c) else ""
    )
    return df


def draw_boxes(img_path, boxes):
    """Draws each box in its class color AND stamps the row index (id) on the box,
    so a box on the image can always be matched back to its table row — essential
    when several boxes are the same class (e.g. three 'Bolt' entries) and look
    identical in color/name alone."""
    im = Image.open(img_path).convert("RGB")
    w, h = im.size
    draw = ImageDraw.Draw(im)

    for i, (cls_id, xc, yc, bw, bh) in enumerate(boxes):
        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w
        y2 = (yc + bh / 2) * h

        color = CLASS_COLORS.get(cls_id, DEFAULT_COLOR)
        name = CLASS_NAMES.get(cls_id, f"UNKNOWN({cls_id})")
        tag = f"#{i} {name}"

        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        text_w = 6 * len(tag) + 6
        draw.rectangle([x1, max(0, y1 - 16), x1 + text_w, max(0, y1 - 16) + 14], fill=color)
        draw.text((x1 + 3, max(0, y1 - 16)), tag, fill="black")

    return im


def _row_count_update(df):
    """Force the Dataframe component's visible row count to match the actual
    number of boxes, every time. Without this, some Gradio versions keep
    whatever row_count was set at component-build time (or a default page
    size) and will not grow to show every box for classes with many
    annotations. Passing 'fixed' with the live length guarantees all rows
    render, however many there are (up to MAX_BOXES_PER_IMAGE)."""
    n = 0 if df is None else len(df)
    n = min(max(n, 1), MAX_BOXES_PER_IMAGE)
    return gr.Dataframe(row_count=(n, "fixed"))


def load_current():
    if not images:
        return None, pd.DataFrame(), "0 / 0", "No images found", _row_count_update(None)
    idx = state["idx"]
    img_path = images[idx]
    boxes = boxes_for_image(img_path)
    im = draw_boxes(img_path, boxes)
    df = boxes_to_df(boxes)
    counter = f"{idx + 1} / {len(images)}  —  {os.path.basename(img_path)}  ({len(boxes)} box(es))"
    if len(boxes) > MAX_BOXES_PER_IMAGE:
        note = (f"⚠️ This image has {len(boxes)} boxes, above the {MAX_BOXES_PER_IMAGE}-box "
                f"display limit — some rows may not render. Raise MAX_BOXES_PER_IMAGE if this recurs.")
    else:
        note = "⚠️ Unsaved edits pending" if img_path in edited_boxes else ""
    return im, df, counter, note, _row_count_update(df)


def next_image():
    if state["idx"] < len(images) - 1:
        state["idx"] += 1
    return load_current()


def prev_image():
    if state["idx"] > 0:
        state["idx"] -= 1
    return load_current()


def preview_edits(df):
    """Redraw the current image using the edited table, without saving.
    Also refreshes the class_name column and re-numbers rows so the #id shown
    on each box always matches its table row — important when several boxes
    share the same class and would otherwise be indistinguishable."""
    if not images:
        return None, pd.DataFrame(), "No images found", _row_count_update(None)
    idx = state["idx"]
    img_path = images[idx]

    boxes = df_to_boxes(df)

    # validate class ids
    bad_ids = sorted({b[0] for b in boxes if b[0] not in VALID_CLASS_IDS})
    if bad_ids:
        warning = f"⚠️ Unknown class id(s) {bad_ids} — allowed: {sorted(VALID_CLASS_IDS)}. Fix before saving."
    else:
        warning = f"✅ Preview updated ({len(boxes)} box(es), not saved yet). Click 'Approve & Save' to write this pair."

    edited_boxes[img_path] = boxes  # remember edit even if it has bad ids, so table doesn't reset
    im = draw_boxes(img_path, boxes)
    df_out = sync_class_names(boxes_to_df(boxes))
    return im, df_out, warning, _row_count_update(df_out)


def save_current(df):
    """Write corrected label file and copy image+label pair to Main Dataset with matching names."""
    if not images:
        return None, pd.DataFrame(), "0 / 0", "Nothing to save", _row_count_update(None)

    idx = state["idx"]
    img_path = images[idx]
    boxes = df_to_boxes(df)

    bad_ids = sorted({b[0] for b in boxes if b[0] not in VALID_CLASS_IDS})
    if bad_ids:
        im = draw_boxes(img_path, boxes)
        df_out = boxes_to_df(boxes)
        counter = f"{idx + 1} / {len(images)}  —  {os.path.basename(img_path)}"
        return im, df_out, counter, f"❌ Not saved — unknown class id(s) {bad_ids}. Fix and try again.", _row_count_update(df_out)

    # Correct, matching filenames: same base name, image keeps its own extension,
    # label is always <base>.txt
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    img_ext = os.path.splitext(img_path)[1]
    dest_img_path = os.path.join(DEST_IMG_DIR, base_name + img_ext)
    dest_lbl_path = os.path.join(DEST_LBL_DIR, base_name + ".txt")

    # write corrected label content — one line per box, however many there are
    lines = [f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}" for cls_id, xc, yc, bw, bh in boxes]
    with open(dest_lbl_path, "w") as f:
        f.write("\n".join(lines) + ("\n" if lines else ""))

    shutil.copy2(img_path, dest_img_path)

    status = f"✅ Saved pair: {os.path.basename(dest_img_path)}  +  {os.path.basename(dest_lbl_path)}  ({len(boxes)} box(es))"

    # clear the pending-edit flag for this image now that it's saved, and move on
    edited_boxes.pop(img_path, None)
    if idx < len(images) - 1:
        state["idx"] += 1

    im, df_out, counter, _note, row_update = load_current()
    return im, df_out, counter, status, row_update


with gr.Blocks(title="Railway Dataset Verification") as demo:
    gr.Markdown("## 🚆 Railway Dataset — Annotation Verification & Class Correction")
    gr.Markdown(
        "Review each image. Every box in the label file gets its own row below — "
        "a class (e.g. **Bolt**, **Wheel**) can have any number of boxes. "
        "If a box's **class number is wrong**, edit the `class_id` column, click "
        "**Preview edits** to see it redrawn, then **Approve & Save** to push the "
        "corrected image + label pair to the Main Dataset."
    )

    legend_html = "<div style='display:flex; flex-wrap:wrap; gap:14px; margin-bottom:8px;'>"
    for cls_id, name in CLASS_NAMES.items():
        r, g, b = CLASS_COLORS[cls_id]
        legend_html += (
            f"<div style='display:flex; align-items:center; gap:6px;'>"
            f"<span style='width:14px; height:14px; border-radius:3px; "
            f"background:rgb({r},{g},{b}); display:inline-block; border:1px solid #333;'></span>"
            f"<span style='font-size:13px;'>{cls_id}: {name}</span></div>"
        )
    legend_html += "</div>"
    gr.HTML(legend_html)

    img_out = gr.Image(label="Image with annotations", type="pil")

    gr.Markdown(
        f"Edit `class_id` below (valid ids: 0-7). `class_name` is just a readable hint. "
        f"One row = one box — classes can repeat across multiple rows "
        f"(up to {MAX_BOXES_PER_IMAGE} boxes total per image)."
    )
    table_out = gr.Dataframe(
        headers=["id", "class_id", "class_name", "x_center", "y_center", "width", "height"],
        datatype=["number", "number", "str", "number", "number", "number", "number"],
        interactive=True,
        row_count=(1, "dynamic"),
        col_count=(7, "fixed"),
        label=f"Boxes for this image (edit class_id, then Preview) — one row per box, up to {MAX_BOXES_PER_IMAGE}",
        wrap=True,
    )

    counter_out = gr.Textbox(label="Progress", interactive=False)
    status_out = gr.Textbox(label="Status", interactive=False)

    with gr.Row():
        prev_btn = gr.Button("⬅ Previous")
        preview_btn = gr.Button("👁 Preview edits")
        save_btn = gr.Button("✅ Approve & Save", variant="primary")
        next_btn = gr.Button("Next ➡")

    demo.load(load_current, outputs=[img_out, table_out, counter_out, status_out, table_out])
    prev_btn.click(prev_image, outputs=[img_out, table_out, counter_out, status_out, table_out])
    next_btn.click(next_image, outputs=[img_out, table_out, counter_out, status_out, table_out])
    preview_btn.click(preview_edits, inputs=[table_out], outputs=[img_out, table_out, status_out, table_out])
    save_btn.click(save_current, inputs=[table_out], outputs=[img_out, table_out, counter_out, status_out, table_out])

demo.launch(share=True, debug=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracted images to /content/Set2
Extracted annotations to /content/Set2an


/usr/local/lib/python3.12/dist-packages/gradio/components/dataframe.py:192: UserWarning: The `col_count` parameter is deprecated and will be removed. Please use `column_count` instead.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2d4ab9253ba79567df.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/components/dataframe.py:192: UserWarning: The `col_count` parameter is deprecated and will be removed. Please use `column_count` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/components/dataframe.py:192: UserWarning: The `col_count` parameter is deprecated and will be removed. Please use `column_count` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCES

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2d4ab9253ba79567df.gradio.live
